## LLM Shape Trace — Manual Step-by-Step Walkthrough
This notebook **manually** tracks tensor shapes as text flows through your GPT-style model — from **input text** → **token IDs** → **embeddings** → **Transformer block** → **logits** → **first generated token** → **decoded text**.

**Note on context length:**  
> In real models the context window is often **C = 1024** (or larger).  
> We keep that as the conceptual setting here.  
> Shapes below show **L ≤ C** at each pass.  
> For readability, when illustrating per-step shapes, we name dimensions instead of expanding huge matrices.

---
## ⚙️ Configuration (Conceptual)

| Symbol | Meaning | Example |
|:--|:--|:--|
| **B** | Batch size | 2 |
| **L₀** | Seed length (initial tokens) | 4 |
| **E** | Embedding dimension | 768 |
| **V** | Vocabulary size | 50 257 |
| **H** | Attention heads | 12 |
| **D** | Head dim = E ÷ H | 64 |
| **C** | Context length | **1024** (typical) |

We walk through **one decode step** (producing the **first new token**) to close the loop from input → output.



---

### 🪄 Step 0 — Input Text → Token IDs (Seed)

```python
# Input texts (B = 2):
["Every effort moves you", "Every effort moves you"]

# Tokenizer output (example IDs):
tok_ids = [
  [15496, 3212, 7325, 345],
  [15496, 3212, 7325, 345]
]
Shape: [B, L₀] == [2, 4] with L₀ ≤ C = 1024
```

### 🧱 Step 1 — Forward Pass to Logits (Predicting the Next Token)

### 1.1 Embeddings

```python
# Token and positional embeddings
tok_emb = self.tok_emb(tok_ids)              # [B, L₀, E]
pos_emb = self.pos_emb(torch.arange(L₀))     # [L₀, E]
x = tok_emb + pos_emb                        # broadcast add
x = self.drop_emb(x)                         # [B, L₀, E]
```

| Operation | Description | **Layer shape** | **Output shape** |
|:--|:--|:--|:--|
| `tok_emb(tok_ids)` | Look up token embeddings from vocabulary | **[V, E] = [50 257, 768]** | **[B, L₀, E] = [2, 4, 768]** |
| `pos_emb(arange(L₀))` | Learnable positional embeddings (up to context length) | **[C, E] = [1024, 768]** | **[L₀, E] = [4, 768]** |
| Add & broadcast | `x = tok_emb + pos_emb` | — | **[B, L₀, E] = [2, 4, 768]** |
| Dropout | (no parameter matrix) | — | **[B, L₀, E] = [2, 4, 768]** |


### 1.2 Transformer Block (Pre-LayerNorm)

#### (a) LayerNorm (pre-attention)
```python
x → norm1(x)
Shape: [2, 4, 768]
```

| Operation | Description | **Layer shape** | **Output shape** |
|:--|:--|:--|:--|
| `LayerNorm` | Normalize across embedding dim | — | **[B, L, E] = [2, 4, 768]** |


#### (b) Multi-Head Self-Attention

```python
ql = nn.Linear(E, E, bias=False)  # [B, L, E]
kl = nn.Linear(E, E, bias=False)  # [B, L, E]
vl = nn.Linear(E, E, bias=False)  # [B, L, E]
Q, K, V = ql(x), kl(x), vl(x)  # each [B, L, E] = [2, 4, 768]
attn_scores = (Q @ K.T) / sqrt(D)
attn_scores.masked_fill_(mask, -inf)
attn_weights = attn_scores.softmax(dim=-1)
context_vec = attn_weights @ V
x = out_proj(context_vec)
```

| Step | Operation | **Layer shape** | **Output shape** |
|:--|:--|:--|:--|
| Linear projections | `Q, K, V = W*x` | each **[E, E] = [768, 768]** | **[B, L, E] = [2, 4, 768]** |
| Split into heads | reshape `[B,L,H,D]` | — | **[2, 4, 12, 64]** |
| Transpose | to `[B,H,L,D]` | — | **[2, 12, 4, 64]** |
| Attention scores | `Q @ Kᵀ` | — | **[2, 12, 4, 4]** |
| Scale | divide by √D (=8) | — | **[2, 12, 4, 4]** |
| Mask | causal mask applied | — | **[2, 12, 4, 4]** |
| Softmax | normalize last dim | — | **[2, 12, 4, 4]** |
| Weighted sum | `attn_weights @ V` | — | **[2, 12, 4, 64]** |
| Transpose back | `[B,L,H,D]` | — | **[2, 4, 12, 64]** |
| Merge heads | reshape `[B,L,E]` | — | **[2, 4, 768]** |
| Output projection | `out_proj` | **[E, E] = [768, 768]** | **[B,L,E] = [2, 4, 768]** |

#### (c) Residual Connection (after attention)
```python
x = x + dropout(attn_output)
Shape: [2, 4, 768]
```

#### (d) LayerNorm (pre-FFN)
```python
x → norm2(x)
Shape: [2, 4, 768]
```

#### (e) Feed-Forward Network (FFN)

```python
x_ffn = self.l1(x)      # 768 → 4 * 768 = 3072
x_ffn = F.gelu(x_ffn)
x_ffn = self.l2(x_ffn)  # 3072 → 768
```

| Step | Operation | Output shape |
|:--|:--|:--|
| Linear 1 | `E → 4E` | **[2, 4, 3072]** |
| GELU | activation | **[2, 4, 3072]** |
| Linear 2 | `4E → E` | **[2, 4, 768]** |

#### (f) Residual Connection (after FFN)
```python
x = x + dropout(ffn_output)
Shape: [2, 4, 768]
```

### 1.3 Final Layers + Projection to Vocab (Logits)

```python
x = self.final_norm(x)          # [B, L, E] = [2, 4, 768]
logits = self.out_head(x)       # 768 → V = 50,257 => [B, L, V] = [2, 4, 50,257]
```

| Operation | Description | Layer shape | Output shape |
|:--|:--|:--|:--|
| Final LayerNorm | Normalize last block output | -- | **[B, L, E] = [2, 4, 768]** |
| Linear Head | project to vocab logits | **[E, V] = [768, 50257]** | **[B, L, V] = [2, 4, 50257]** |

✅ **Final model output (logits)**: `[B=2, L=4, V=50,257]`

→ Take the last position’s logits (logits[:, −1, :] → [2, 50 257]) to predict the next token.

### ▶️ Step 2 — Read Last-Step Logits → Sample Next Token → Append

```python
logits_last = logits[:, -1, :]                           # [B, V]
probs = torch.softmax(logits_last, dim=-1)               # [B, V]
next_tok_id = torch.argmax(probs, dim=-1, keepdim=True)  # [B, 1]
tok_ids = torch.cat((tok_ids, next_tok_id), dim=1)       # [B, L₀+1]
```

| Operation | Description | Output shape |
|:--|:--|:--|
| Select last logits | logits[:, -1, :] | **[B,V] = [2, 50 257]** |
| Softmax | convert to probabilities | **[B,V] = [2, 50 257]** |
| Argmax / Sample | pick next token ID | **[B,1] = [2, 1]** |
| Concatenate | append to token sequence | **[B,L₀+1] = [2, 5]** |

✅ You’ve now generated the first new token.

### 🗣️ Step 3 — Decode to Text

```python
decoded_text = tokenizer.decode(tok_ids[i])
```

Result: Original sentence + the newly generated token.

**Example:**  
"Every effort moves you forward"  
"Every effort moves you forward"


### Quick Recap of Shapes (One Pass + One New Token)

| Stage | Operation | Shape |
|:--|:--|:--|
| Tokenization | text → IDs | **[B, L₀]** |
| Embeddings | IDs → vectors | **[B, L₀, E]** |
| Attention | context mixing | **[B, L₀, E]** |
| FFN | non-linear transform | **[B, L₀, E]** |
| Head | project to vocab | **[B, L, V]** |
| Next token | read last logits → sample | **[B, 1]** |
| Append | concat to sequence | **[B, L + 1]** |